# Script 03 · Exploración de imágenes satelitales

**Curso:** Introducción a Google Earth Engine  
**Autora:** Grettel Vargas Azofeifa  
**Modalidad:** material de apoyo para GitHub y Google Earth Engine

> Los bloques de código están escritos en JavaScript para ejecutarse en el Editor de código de Google Earth Engine.

Aprenda a buscar, filtrar, explorar y visualizar imágenes Sentinel-2 utilizando el área de interés definida en el Script 02.

## 🎯 Objetivos

1. Recuperar y reutilizar un área de interés.
2. Cargar una colección Sentinel-2.
3. Filtrar imágenes por área, fecha y nubosidad.
4. Explorar propiedades y bandas de una imagen.
5. Crear una composición mediana.
6. Visualizar color natural, falso color y una composición agrícola.
7. Comparar una composición con una imagen individual.
8. Cargar Landsat 8 y Landsat 9.
9. Crear composiciones Landsat para agricultura, falso color y color natural.
10. Construir una comparación Split sincronizada.

> **Nota:** Conceptos: ee.ImageCollection , filterBounds() , filterDate() , ee.Filter.lte() , first() , sort() , median() , clip() , bandNames() , factores de escala Landsat, combinaciones de bandas y ui.SplitPanel .

## 📖 Cómo usar esta guía

- Ejecute cada sección en el orden indicado.
- Revise la cantidad de imágenes en la Consola .
- Active y desactive las capas desde Layers .
- Compare visualmente las composiciones de bandas.
- Modifique fechas y nubosidad únicamente después de ejecutar el ejemplo original.

## 1. Recuperar el área de interés

Se cargan las provincias de Costa Rica y se selecciona una de ellas como área de interés para el análisis de imágenes.

In [ ]:
var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq('ADM0_NAME', 'Costa Rica')
);

var nombreProvincia = 'Guanacaste';

var provinciaSeleccionada = provincias.filter(
  ee.Filter.eq('ADM1_NAME', nombreProvincia)
);

var areaInteres =
  provinciaSeleccionada.geometry();

Map.centerObject(areaInteres, 8);
Map.setOptions('HYBRID');


> **Práctica:** 🧪 Práctica: cambie 'Guanacaste' por otra provincia y ejecute nuevamente.

## 2. Cargar la colección Sentinel-2

Se utilizará la colección armonizada de reflectancia de superficie de Sentinel-2.

In [ ]:
var sentinel2 = ee.ImageCollection(
  'COPERNICUS/S2_SR_HARMONIZED'
);

print(
  'Colección Sentinel-2:',
  sentinel2
);


> **Nota:** Una ImageCollection contiene múltiples imágenes capturadas en fechas y condiciones diferentes.

## 3. Definir fechas y nubosidad

Las variables permiten cambiar fácilmente el periodo de análisis y el porcentaje máximo de nubosidad aceptado.

In [ ]:
var fechaInicio = '2025-01-01';
var fechaFin = '2025-12-31';
var nubosidadMaxima = 20;


## 4. Aplicar filtros

La colección se filtra por área de interés, periodo de tiempo y nubosidad de la escena.

In [ ]:
var sentinelFiltrado = sentinel2
  .filterBounds(areaInteres)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  );

print(
  'Colección filtrada:',
  sentinelFiltrado
);

print(
  'Cantidad de imágenes:',
  sentinelFiltrado.size()
);


> **Nota:** Importante: CLOUDY_PIXEL_PERCENTAGE describe la nubosidad estimada de la escena completa; todavía no elimina las nubes píxel por píxel.

## 5. Explorar una imagen

Se revisa la primera imagen de la colección para conocer sus bandas, propiedades y fecha de adquisición.

In [ ]:
var primeraImagen =
  sentinelFiltrado.first();

print(
  'Primera imagen:',
  primeraImagen
);

print(
  'Bandas disponibles:',
  primeraImagen.bandNames()
);

print(
  'Fecha de la primera imagen:',
  ee.Date(
    primeraImagen.get(
      'system:time_start'
    )
  ).format('YYYY-MM-dd')
);


## 6. Crear una composición mediana

La mediana combina todas las imágenes filtradas y utiliza el valor mediano de cada píxel. Esto ayuda a reducir valores extremos y parte de la nubosidad.

In [ ]:
var composicionMediana =
  sentinelFiltrado
    .median()
    .clip(areaInteres);

print(
  'Composición mediana:',
  composicionMediana
);


## 7. Visualizar color natural

La combinación B4, B3 y B2 representa rojo, verde y azul de forma similar a la visión humana.

In [ ]:
var colorNatural = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

Map.addLayer(
  composicionMediana,
  colorNatural,
  'Sentinel-2 · Color natural'
);


## 8. Visualizar falso color

La combinación B8, B4 y B3 destaca la vegetación en tonos rojos y facilita su interpretación.

In [ ]:
var falsoColor = {
  bands: ['B8', 'B4', 'B3'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

Map.addLayer(
  composicionMediana,
  falsoColor,
  'Sentinel-2 · Falso color',
  false
);


## 9. Visualizar una composición agrícola

La combinación B11, B8 y B2 permite resaltar diferencias de humedad, vegetación y suelo.

In [ ]:
var agricultura = {
  bands: ['B11', 'B8', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.15
};

Map.addLayer(
  composicionMediana,
  agricultura,
  'Sentinel-2 · Agricultura',
  false
);


## 10. Añadir el límite del área de interés

El límite administrativo se coloca sobre las imágenes para facilitar la orientación espacial.

In [ ]:
Map.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 3
  }),
  {},
  'Límite del área de interés'
);


## 11. Seleccionar la imagen menos nubosa

La colección se ordena por nubosidad para seleccionar la escena con el menor porcentaje registrado.

In [ ]:
var imagenMenosNubosa = ee.Image(
  sentinelFiltrado.sort(
    'CLOUDY_PIXEL_PERCENTAGE'
  ).first()
).clip(areaInteres);

print(
  'Imagen con menor nubosidad:',
  imagenMenosNubosa
);

print(
  'Nubosidad:',
  imagenMenosNubosa.get(
    'CLOUDY_PIXEL_PERCENTAGE'
  )
);

Map.addLayer(
  imagenMenosNubosa,
  colorNatural,
  'Imagen individual menos nubosa',
  false
);


## 12. Cargar Landsat 8 y Landsat 9

Para ampliar la comparación se combinan las colecciones Landsat 8 y Landsat 9 de reflectancia de superficie. Ambas utilizan nombres de bandas equivalentes.

In [ ]:
// Función para aplicar los factores de escala.
function escalarLandsat(imagen) {
  var bandasOpticas = imagen
    .select('SR_B.')
    .multiply(0.0000275)
    .add(-0.2);

  return imagen
    .addBands(
      bandasOpticas,
      null,
      true
    );
}

// Cargar Landsat 8.
var landsat8 = ee.ImageCollection(
  'LANDSAT/LC08/C02/T1_L2'
);

// Cargar Landsat 9.
var landsat9 = ee.ImageCollection(
  'LANDSAT/LC09/C02/T1_L2'
);

// Unir ambas colecciones.
var landsat = landsat8
  .merge(landsat9)
  .filterBounds(areaInteres)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUD_COVER',
      nubosidadMaxima
    )
  )
  .map(escalarLandsat);

print(
  'Colección Landsat 8 y 9:',
  landsat
);

print(
  'Cantidad de imágenes Landsat:',
  landsat.size()
);


> **Nota:** Importante: las bandas de reflectancia de superficie Landsat Collection 2 Level 2 requieren aplicar factores de escala antes de visualizarlas correctamente.

## 13. Crear composiciones Landsat

Se genera una composición mediana Landsat y se visualiza en color natural, falso color y una combinación útil para aplicaciones agrícolas.

In [ ]:
var landsatMediana = landsat
  .median()
  .clip(areaInteres);

// Color natural: rojo, verde y azul.
var landsatNatural = {
  bands: [
    'SR_B4',
    'SR_B3',
    'SR_B2'
  ],
  min: 0,
  max: 0.3,
  gamma: 1.2
};

// Falso color: infrarrojo cercano, rojo y verde.
var landsatFalsoColor = {
  bands: [
    'SR_B5',
    'SR_B4',
    'SR_B3'
  ],
  min: 0,
  max: 0.4,
  gamma: 1.2
};

// Agricultura: SWIR1, NIR y azul.
var landsatAgricultura = {
  bands: [
    'SR_B6',
    'SR_B5',
    'SR_B2'
  ],
  min: 0,
  max: 0.4,
  gamma: 1.15
};

Map.addLayer(
  landsatMediana,
  landsatNatural,
  'Landsat · Color natural',
  false
);

Map.addLayer(
  landsatMediana,
  landsatFalsoColor,
  'Landsat · Falso color',
  false
);

Map.addLayer(
  landsatMediana,
  landsatAgricultura,
  'Landsat · Agricultura',
  false
);


> **Práctica:** 🧪 Práctica: compare las composiciones Landsat con las equivalentes de Sentinel-2. Observe diferencias de resolución espacial, detalle y color.

## 14. Crear una comparación Split

El panel dividido permite comparar dos imágenes sincronizadas. En este ejemplo se muestra Sentinel-2 a la izquierda y Landsat a la derecha.

In [ ]:
// Crear dos mapas independientes.
var mapaIzquierdo = ui.Map();
var mapaDerecho = ui.Map();

// Añadir Sentinel-2 al mapa izquierdo.
mapaIzquierdo.addLayer(
  composicionMediana,
  colorNatural,
  'Sentinel-2 · Natural'
);

// Añadir Landsat al mapa derecho.
mapaDerecho.addLayer(
  landsatMediana,
  landsatNatural,
  'Landsat · Natural'
);

// Añadir el límite en ambos mapas.
mapaIzquierdo.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 2
  }),
  {},
  'Límite'
);

mapaDerecho.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 2
  }),
  {},
  'Límite'
);

// Sincronizar el movimiento de los mapas.
var enlaceMapas = ui.Map.Linker([
  mapaIzquierdo,
  mapaDerecho
]);

// Centrar ambos mapas.
mapaIzquierdo.centerObject(
  areaInteres,
  9
);

// Crear el panel dividido.
var panelComparacion = ui.SplitPanel({
  firstPanel: mapaIzquierdo,
  secondPanel: mapaDerecho,
  orientation: 'horizontal',
  wipe: true
});

// Reemplazar la interfaz principal.
ui.root.widgets().reset([
  panelComparacion
]);


> **Nota:** Importante: ui.root.widgets().reset() reemplaza temporalmente la interfaz estándar del Code Editor por el panel comparativo. Ejecute este bloque al final del script.

> **Práctica:** 🧪 Práctica: cambie las capas del Split para comparar falso color Sentinel-2 con falso color Landsat.

## 15. Comparar los resultados

1. Active únicamente la composición mediana en color natural.
2. Active la imagen individual menos nubosa.
3. Observe nubes, sombras, diferencias de color y continuidad espacial.
4. Compare después el falso color y la composición agrícola.
5. Utilice el panel Split para comparar Sentinel-2 y Landsat.

## 🚀 Desafío opcional

Modifique el periodo de análisis y el porcentaje máximo de nubosidad. Compare cómo cambia la cantidad de imágenes y la composición resultante.

In [ ]:
var fechaInicio = '2025-01-01';
var fechaFin = '2025-04-30';
var nubosidadMaxima = 10;


## ✅ Resumen

- Definimos un área de interés.
- Cargamos Sentinel-2.
- Filtramos por ubicación, fecha y nubosidad.
- Exploramos propiedades y bandas.
- Creamos una composición mediana.
- Visualizamos color natural, falso color y agricultura.
- Seleccionamos la imagen menos nubosa.
- Comparamos una composición con una escena individual.
- Cargamos y combinamos Landsat 8 y Landsat 9.
- Visualizamos Landsat en color natural, falso color y agricultura.
- Creamos un panel Split para comparar Sentinel-2 y Landsat.

> **Nota:** Siguiente paso: en el Script 04 se utilizarán estas bandas para calcular NDVI, EVI, NDWI y NBR.

## 💻 Código completo

Abra esta sección para consultar o copiar el script íntegro.

In [ ]:
// ================================================================
// CURSO INTRODUCTORIO GOOGLE EARTH ENGINE
// SCRIPT 03 — EXPLORACIÓN DE IMÁGENES SATELITALES
// Autora: Grettel Vargas Azofeifa
// ================================================================

// 1. CARGAR LAS PROVINCIAS DE COSTA RICA
var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq('ADM0_NAME', 'Costa Rica')
);

// 2. DEFINIR UNA PROVINCIA COMO ÁREA DE INTERÉS
var nombreProvincia = 'Guanacaste';

var provinciaSeleccionada = provincias.filter(
  ee.Filter.eq('ADM1_NAME', nombreProvincia)
);

var areaInteres = provinciaSeleccionada.geometry();

print('Área de interés:', areaInteres);

Map.centerObject(areaInteres, 8);
Map.setOptions('HYBRID');

// 3. CARGAR SENTINEL-2
var sentinel2 = ee.ImageCollection(
  'COPERNICUS/S2_SR_HARMONIZED'
);

print('Colección Sentinel-2:', sentinel2);

// 4. DEFINIR FECHAS Y NUBOSIDAD
var fechaInicio = '2025-01-01';
var fechaFin = '2025-12-31';
var nubosidadMaxima = 20;

// 5. FILTRAR LA COLECCIÓN
var sentinelFiltrado = sentinel2
  .filterBounds(areaInteres)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  );

print('Colección filtrada:', sentinelFiltrado);
print('Cantidad de imágenes:', sentinelFiltrado.size());

// 6. REVISAR LA PRIMERA IMAGEN
var primeraImagen = sentinelFiltrado.first();

print('Primera imagen:', primeraImagen);
print('Bandas disponibles:', primeraImagen.bandNames());

print(
  'Fecha de la primera imagen:',
  ee.Date(
    primeraImagen.get('system:time_start')
  ).format('YYYY-MM-dd')
);

// 7. CREAR UNA COMPOSICIÓN MEDIANA
var composicionMediana = sentinelFiltrado
  .median()
  .clip(areaInteres);

print('Composición mediana:', composicionMediana);

// 8. VISUALIZAR COLOR NATURAL
var colorNatural = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

Map.addLayer(
  composicionMediana,
  colorNatural,
  'Sentinel-2 · Color natural'
);

// 9. VISUALIZAR FALSO COLOR
var falsoColor = {
  bands: ['B8', 'B4', 'B3'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

Map.addLayer(
  composicionMediana,
  falsoColor,
  'Sentinel-2 · Falso color',
  false
);

// 10. VISUALIZAR AGRICULTURA
var agricultura = {
  bands: ['B11', 'B8', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.15
};

Map.addLayer(
  composicionMediana,
  agricultura,
  'Sentinel-2 · Agricultura',
  false
);

// 11. AÑADIR EL LÍMITE DEL ÁREA DE INTERÉS
Map.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 3
  }),
  {},
  'Límite del área de interés'
);

// 12. COMPARAR UNA IMAGEN INDIVIDUAL
var imagenMenosNubosa = ee.Image(
  sentinelFiltrado.sort(
    'CLOUDY_PIXEL_PERCENTAGE'
  ).first()
).clip(areaInteres);

print('Imagen con menor nubosidad:', imagenMenosNubosa);

Map.addLayer(
  imagenMenosNubosa,
  colorNatural,
  'Imagen individual menos nubosa',
  false
);

// 13. MOSTRAR INFORMACIÓN DE NUBOSIDAD
print(
  'Nubosidad de la imagen seleccionada:',
  imagenMenosNubosa.get(
    'CLOUDY_PIXEL_PERCENTAGE'
  )
);

// 14. CARGAR Y PREPARAR LANDSAT 8 Y 9
function escalarLandsat(imagen) {
  var bandasOpticas = imagen
    .select('SR_B.')
    .multiply(0.0000275)
    .add(-0.2);

  return imagen.addBands(
    bandasOpticas,
    null,
    true
  );
}

var landsat8 = ee.ImageCollection(
  'LANDSAT/LC08/C02/T1_L2'
);

var landsat9 = ee.ImageCollection(
  'LANDSAT/LC09/C02/T1_L2'
);

var landsat = landsat8
  .merge(landsat9)
  .filterBounds(areaInteres)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUD_COVER',
      nubosidadMaxima
    )
  )
  .map(escalarLandsat);

print('Colección Landsat 8 y 9:', landsat);
print('Cantidad de imágenes Landsat:', landsat.size());

// 15. CREAR COMPOSICIÓN MEDIANA LANDSAT
var landsatMediana = landsat
  .median()
  .clip(areaInteres);

var landsatNatural = {
  bands: ['SR_B4', 'SR_B3', 'SR_B2'],
  min: 0,
  max: 0.3,
  gamma: 1.2
};

var landsatFalsoColor = {
  bands: ['SR_B5', 'SR_B4', 'SR_B3'],
  min: 0,
  max: 0.4,
  gamma: 1.2
};

var landsatAgricultura = {
  bands: ['SR_B6', 'SR_B5', 'SR_B2'],
  min: 0,
  max: 0.4,
  gamma: 1.15
};

Map.addLayer(
  landsatMediana,
  landsatNatural,
  'Landsat · Color natural',
  false
);

Map.addLayer(
  landsatMediana,
  landsatFalsoColor,
  'Landsat · Falso color',
  false
);

Map.addLayer(
  landsatMediana,
  landsatAgricultura,
  'Landsat · Agricultura',
  false
);

// 16. CREAR COMPARACIÓN SPLIT
var mapaIzquierdo = ui.Map();
var mapaDerecho = ui.Map();

mapaIzquierdo.addLayer(
  composicionMediana,
  colorNatural,
  'Sentinel-2 · Natural'
);

mapaDerecho.addLayer(
  landsatMediana,
  landsatNatural,
  'Landsat · Natural'
);

mapaIzquierdo.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 2
  }),
  {},
  'Límite'
);

mapaDerecho.addLayer(
  provinciaSeleccionada.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 2
  }),
  {},
  'Límite'
);

var enlaceMapas = ui.Map.Linker([
  mapaIzquierdo,
  mapaDerecho
]);

mapaIzquierdo.centerObject(
  areaInteres,
  9
);

var panelComparacion = ui.SplitPanel({
  firstPanel: mapaIzquierdo,
  secondPanel: mapaDerecho,
  orientation: 'horizontal',
  wipe: true
});

ui.root.widgets().reset([
  panelComparacion
]);

// 17. RESUMEN
print(
  'Resumen:',
  'Se filtró Sentinel-2 por área, fecha y nubosidad; ' +
  'se creó una composición mediana y se visualizaron ' +
  'distintas combinaciones de bandas, además de comparar ' +
  'Sentinel-2 y Landsat mediante un panel Split.'
);


## 📚 Recursos oficiales

- Creación y manejo de ImageCollection
- Filtros de colecciones
- Reducción de ImageCollection
- Sentinel-2 Surface Reflectance Harmonized
- ee.Image.clip()
- Map.addLayer()
- Landsat 8 Collection 2 Level 2
- Landsat 9 Collection 2 Level 2
- ui.SplitPanel
- ui.Map.Linker

## 🧭 Continuar el curso